# Laboratory: Exploratory Data Analysis, PCA and Kernel PCA

**Case study:** Cardiotocography (CTG) — fetal monitoring measurements.  
**Learning goals:** inspect real multivariate data, identify correlated features, interpret linear principal components, explore nonlinear embeddings, and evaluate dimensionality reduction without data leakage.

The UCI Cardiotocography dataset contains 2,126 recordings and 21 numerical input measurements. We use the **NSP** label (1 = normal, 2 = suspect, 3 = pathological) only for **coloring plots and evaluating classification**. PCA and Kernel PCA must not use target labels during fitting.



### Setup and data loading

Install the optional dataset dependency once (`%pip install ucimlrepo`). We load data directly from the UCI repository; an internet connection is required the first time. Use the exact same dataset in the exercises and the solution.

https://archive.ics.uci.edu/datasets

In [11]:
# Uncomment if necessary:
# %pip install ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo
from sklearn.decomposition import PCA, KernelPCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

ctg = fetch_ucirepo(id=193)
X = ctg.data.features.copy()
y = ctg.data.targets["NSP"].copy().astype(int)
class_names = {1: "Normal", 2: "Suspect", 3: "Pathological"}

display(X.head())
display(y.head())

,LB,AC,FM,UC,DL,DS,DP,ASTV,MSTV,ALTV,...,Width,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance,Tendency
0,120,0.000,0.0,0.000,0.000,0.0,0.0,73,0.5,43,...,64,62,126,2,0,120,137,121,73,1
1,132,0.006,0.0,0.006,0.003,0.0,0.0,17,2.1,0,...,130,68,198,6,1,141,136,140,12,0
2,133,0.003,0.0,0.008,0.003,0.0,0.0,16,2.1,0,...,130,68,198,5,1,141,135,138,13,0
3,134,0.003,0.0,0.008,0.003,0.0,0.0,16,2.4,0,...,117,53,170,11,0,137,134,137,13,1
4,132,0.007,0.0,0.008,0.000,0.0,0.0,16,2.4,0,...,117,53,170,9,0,137,136,138,11,1


0    2
1    1
2    1
3    1
4    1
Name: NSP, dtype: int64

### 1. EDA: data quality and class imbalance

**Tasks**
1. Inspect feature types, missingness, duplicates, and descriptive statistics.
2. Plot the distribution of the three classes. Is this dataset balanced?
3. Choose **four** numerical features, plot their histograms and boxplots (colored by NSP).
4. Write down two observations. Do you see outliers, skewed variables, or differences among classes?

**Interpretation:** Duplicate measurements are not necessarily duplicate patients; report them but do not drop them automatically.

In [3]:
# TODO: inspect X.info(), X.isna(), X.duplicated(), X.describe()
# TODO: plot y.value_counts() in a bar chart
# TODO: choose 4 columns and plot histograms + boxplots

selected_features = list(X.columns[:4])  # Change if you wish
# YOUR CODE HERE

### 2. Relationships and correlations

**Tasks**
1. Plot a pairplot using a subset of ~4–5 features (not all 21: the full plot would be unwieldy).
2. Compute the Pearson correlation matrix and show it as a heatmap.
3. Find the **five strongest absolute off-diagonal correlations** and report their signs.
4. Predict whether PCA can compress the input effectively. Explain your reasoning.

**Question:** Does correlation imply that one measurement causes another?

In [4]:
# TODO: create a pairplot on your selected features with hue="NSP"
# TODO: compute and visualize the correlation matrix
# TODO: extract the strongest distinct correlations (no diagonal or duplicates)
# YOUR CODE HERE

### 3. Standardization and linear PCA

**Tasks**
1. Impute missing values using the feature median (if any), then standardize each feature.
2. Fit PCA with all components. Plot explained variance and cumulative explained variance.
3. Find the minimum number of components needed to explain **80%, 90%, and 95%** of the variance.
4. Plot the PC1–PC2 scores with points colored by NSP. Optionally plot PC1–PC3 in 3D.
5. Compute the loadings for PC1 and PC2: which original features have the largest **absolute** coefficients?

**Questions:** Why standardize? Does a good-looking class separation necessarily follow from large explained variance?

In [5]:
# TODO: SimpleImputer(strategy="median") -> StandardScaler -> PCA
# TODO: scree plot, cumulative explained variance, thresholds
# TODO: 2D scores colored by class, loading bar chart
# YOUR CODE HERE

### 4. PCA reconstruction

Reconstruct the standardized data from 1, 2, 3, 5, 10, and 15 PCs, then compute mean squared reconstruction error.

**Questions:** Is the error monotonic? Why is this a property of PCA? Why is reconstruction quality different from class-prediction quality?

In [6]:
# TODO: fit PCA(n_components=k) for k in [1,2,3,5,10,15]
# TODO: inverse_transform, compute mean squared reconstruction error
# YOUR CODE HERE

### 5. Kernel PCA: do nonlinear components help?

1. Compare **linear PCA** with **RBF Kernel PCA** on the *same standardized observations*.
2. Try `gamma` in `[0.001, 0.01, 0.1, 1.0]`, plotting each two-dimensional embedding with NSP colors.
3. Describe what changes as gamma increases. Which representation *looks* more class-separating?

**Important:** The axes of Kernel PCA are nonlinear components; **PCA explained-variance ratios are not directly available or comparable** for RBF Kernel PCA. A visually separated training embedding is not enough to claim predictive improvement.

**Optional:** Repeat the experiment on the synthetic `make_circles` dataset from the original notebook to see a case where nonlinearity has a clear geometric effect.

In [7]:
# TODO: compare 2D PCA with RBF Kernel PCA for 4 gamma values
# Avoid assuming that a larger gamma is better.
# YOUR CODE HERE

### 6. Fair predictive comparison (no leakage)

Compare a logistic regression classifier on:
- standardized original features;
- PCA with 2 and 5 components;
- Kernel PCA (RBF) with 2 and 5 components.

Use **stratified 5-fold cross-validation** and report **balanced accuracy** (appropriate for imbalanced classes), including mean and standard deviation. Keep median imputation, standardization, PCA/Kernel PCA, and classification **inside each pipeline** so preprocessing is fitted only on training folds.

**Questions:** Does Kernel PCA always outperform PCA? Is the result consistent with your visual impressions? Can you safely choose gamma using the same CV results you report as a final unbiased evaluation? (Discuss nested CV or a held-out test set.)

In [8]:
# TODO: build sklearn pipelines and use cross_validate / StratifiedKFold
# TODO: report mean and standard deviation of test balanced accuracy
# YOUR CODE HERE

### 7. Final report 

Answer:
- Which feature correlations were the strongest?
- How many PCs retain 90% of the variance?
- What do PC1 and PC2 seem to capture?
- What happened when gamma changed?
- Which method had the strongest cross-validated balanced accuracy? What limitations remain?
